# CS781 – Merged Phase 1–4 (4 Phase-1 backbones)

| Backbone | Typical `.pt` | Arch |
|----------|---------------|------|
| `custom_mha` | `legal_lm_custom_mha.pt` | RoPE + MHA + SwiGLU |
| `mla_moe` | `legal_lm_mla_moe.pt` | MLA + MoE |
| `gqa_conv` | `legal_lm_gqa_conv.pt` | RoPE + GQA + Dual-path FFN |
| `gqa_alibi` | `legal_lm_alibi_gqa.pt` | **ALiBi + GQA** |

**Phase 1 = load only.** Set `BACKBONE` and paths; Phase 2/3/4 train heads on that backbone.

### Why different tokenizers?

Each Phase-1 model was trained with its own byte-level BPE (different merge tables → different vocab sizes).  
Using a single shared tokenizer would either:
- produce out-of-range token ids (CUDA assert / silent corruption), or
- feed the model a completely different tokenisation than the one it saw during pre-training.

So we keep **one tokenizer file per backbone** (`.pkl` or `.json`) and load the matching one whenever we switch `BACKBONE`. The model’s embedding matrix size is still taken from the checkpoint; we only assert that it matches the tokenizer’s `vocab_size`.


In [ ]:
!pip -q install ijson tqdm

In [ ]:
# !mkdir -p /kaggle/working/phase1_models
# !cp -v /kaggle/input/datasets/vidya6364/phase1-models/* /kaggle/working/phase1_models/
# !cp -v /kaggle/input/datasets/vidya6364/phase1-models-1k/* /kaggle/working/phase1_models/
# !ls /kaggle/input/

In [ ]:
import os, sys, re, gc, json, math, time, random, hashlib, unicodedata, zipfile
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# ste the seeds for reproducibilty
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE, "| amp:", AMP_DTYPE if USE_AMP else None)

device: cpu | amp: None


## 0. Paths & global config

Edit `DATASET_DIR`, `BACKBONE`, and the four entries in `TOK_PATHS`.  
Everything else is derived from those.


In [ ]:
!ls

cs781_phase7  phase1_models  phase7  state.db


In [ ]:
# ── CHANGE THESE ────────────────────────────────────────────────────────────
DATASET_DIR = "/kaggle/working/phase1_models"   # folder with .pt + tokenizers
BACKBONE    = "mla_moe"   # "custom_mha" | "mla_moe" | "gqa_conv" | "gqa_alibi"

RUN_PHASE2_TRAIN = True
RUN_PHASE3_TRAIN = True

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    login(token=HF_TOKEN)
    print("HF login OK")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("HF login skipped:", e)

HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
P2_OUT  = "/kaggle/working/phase2_lner" if os.path.isdir("/kaggle/working") else "./phase2_lner"
P3_OUT  = "/kaggle/working/phase3" if os.path.isdir("/kaggle/working") else "./phase3"
P4_BASE = "/kaggle/working/phase4" if os.path.isdir("/kaggle/working") else "./phase4"
os.makedirs(P2_OUT, exist_ok=True)
os.makedirs(P3_OUT, exist_ok=True)
os.makedirs(P4_BASE, exist_ok=True)

def _first(*cands):
    """Return the first path that actually exists on disk."""
    for p in cands:
        if p and os.path.exists(p):
            return p
    return cands[0]

# ── Per-backbone tokenizer paths ───────────────────────────────────────────
# Drop your four .pkl (or .json) files into DATASET_DIR and adjust the names
# below if they differ.  The loader accepts both formats.
TOK_PATHS = {
    "custom_mha": _first(
        os.path.join(DATASET_DIR, "legal_bpe_custom_mha.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),          # legacy fallback
    ),
    "mla_moe": _first(
        os.path.join(DATASET_DIR, "legal_bpe_mla_moe.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
    "gqa_conv": _first(
        os.path.join(DATASET_DIR, "legal_bpe_gqa_conv.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
    "gqa_alibi": _first(
        os.path.join(DATASET_DIR, "legal_bpe_tokenizer_gqa_alibi.pkl"),
        os.path.join(DATASET_DIR, "legal_bpe_alibi_gqa.pkl"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
}

TOKENIZER_PATH = TOK_PATHS[BACKBONE]

# ── Checkpoints ────────────────────────────────────────────────────────────
CKPT_V2 = _first(
    os.path.join(DATASET_DIR, "legal_lm_custom_mha.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_custom_mha.pt"),
)
CKPT_V3 = _first(
    os.path.join(DATASET_DIR, "legal_lm_mla_moe_1k.pt"),
    os.path.join(DATASET_DIR, "legal_lm_mla_moe.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_mla_moe.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_mla_moe.pt"),
)
CKPT_GQA = _first(
    os.path.join(DATASET_DIR, "legal_lm_gqa_conv.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_gqa_conv.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_gqa_conv.pt"),
)
CKPT_ALIBI = _first(
    os.path.join(DATASET_DIR, "legal_lm_alibi_gqa.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_alibi_gqa.pt"),
    os.path.join(DATASET_DIR, "legal_lm_alibi.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_alibi_gqa.pt"),
)

CKPTS = {"custom_mha": CKPT_V2, "mla_moe": CKPT_V3, "gqa_conv": CKPT_GQA, "gqa_alibi": CKPT_ALIBI}
PHASE1_CKPT = CKPTS[BACKBONE]

print("BACKBONE     :", BACKBONE)
print("Tokenizer    :", TOKENIZER_PATH, os.path.exists(TOKENIZER_PATH))
for k, v in CKPTS.items():
    print(f"  {k:12s}:", v, os.path.exists(v))
print("Phase-1 used :", PHASE1_CKPT, os.path.exists(PHASE1_CKPT))
for k, p in TOK_PATHS.items():
    print(f"  tok[{k:12s}]:", p, os.path.exists(p))


HF login OK
BACKBONE     : mla_moe
Tokenizer    : /kaggle/working/phase1_models/legal_bpe_mla_moe.json True
  custom_mha  : /kaggle/working/phase1_models/legal_lm_custom_mha.pt True
  mla_moe     : /kaggle/working/phase1_models/legal_lm_mla_moe_1k.pt True
  gqa_conv    : /kaggle/working/phase1_models/legal_lm_gqa_conv.pt True
  gqa_alibi   : /kaggle/working/phase1_models/legal_lm_alibi_gqa.pt True
Phase-1 used : /kaggle/working/phase1_models/legal_lm_mla_moe_1k.pt True
  tok[custom_mha  ]: /kaggle/working/phase1_models/legal_bpe_custom_mha.json True
  tok[mla_moe     ]: /kaggle/working/phase1_models/legal_bpe_mla_moe.json True
  tok[gqa_conv    ]: /kaggle/working/phase1_models/legal_bpe_gqa_conv.json True
  tok[gqa_alibi   ]: /kaggle/working/phase1_models/legal_bpe_tokenizer_gqa_alibi.pkl True


## 1. Shared text cleaning + byte-level BPE tokenizer

The same cleaning pipeline is used for every backbone.  
Only the **merge table / vocab** changes with `BACKBONE`.


In [ ]:
_QUOTES = str.maketrans({"“": '"', "”": '"', "„": '"', "‘": "'", "’": "'", "‚": "'", "`": "'", "´": "'",
                         "–": "-", "—": "-", "−": "-", "\u00ad": ""})
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_WS = re.compile(r"[ \t\u00a0\u1680\u2000-\u200b\u202f\u205f\u3000\ufeff]+")
_DOTS = re.compile(r"\.{4,}|(?:\. ){4,}")
_RULE = re.compile(r"[_\-=*~#]{3,}")
_PAGE = re.compile(r"(?mi)^[ \t]*(?:page\s+)?\d{1,4}[ \t]*$\n?")
_HYPH = re.compile(r"(?<=[a-z])-\n(?=[a-z])")
_WRAP = re.compile(r"(?<=[a-z,;])\n(?=[a-z(])")

def clean_text(t):
    """Light, conservative normalisation of legal text."""
    t = unicodedata.normalize("NFKC", str(t)).replace("\r\n", "\n").replace("\r", "\n")
    t = t.translate(_QUOTES)
    t = _CTRL.sub(" ", t)
    t = _HYPH.sub("", t)
    t = _DOTS.sub("...", t)
    t = _RULE.sub(" ", t)
    t = _PAGE.sub("", t)
    t = _WS.sub(" ", t)
    t = re.sub(r" *\n *", "\n", t)
    t = _WRAP.sub(" ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


class BPETokenizer:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    _split = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+")

    def __init__(self, vocab_size=12000):
        self.vocab_size = vocab_size
        self.merges = []
        self._reset()

    def _reset(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        self._cache = {}

    def _build(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        seen = set(self.itos[4:])
        for a, b in self.merges:
            m = a + b
            if m not in seen:
                seen.add(m); self.itos.append(m)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.ranks = {pair: r for r, pair in enumerate(self.merges)}
        self.merged = {pair: pair[0] + pair[1] for pair in self.merges}
        self._cache = {}
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.vocab_size = len(self.itos)

    def _word(self, w):
        r = self._cache.get(w)
        if r is not None: return r
        toks = [bytes([b]) for b in w.encode("utf-8")]
        while len(toks) > 1:
            best_i, best_r = -1, 1 << 60
            for i in range(len(toks) - 1):
                r_ = self.ranks.get((toks[i], toks[i + 1]), 1 << 60)
                if r_ < best_r: best_i, best_r = i, r_
            if best_i < 0: break
            toks[best_i:best_i + 2] = [toks[best_i] + toks[best_i + 1]]
        r = [self.stoi.get(t, self.unk_id) for t in toks]
        if len(self._cache) > 500_000: self._cache.clear()
        self._cache[w] = r
        return r

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        for w in self._split.findall(text):
            ids.extend(self._word(w))
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        if isinstance(ids, torch.Tensor): ids = ids.tolist()
        parts = []
        for i in ids:
            if i < 4: continue
            t = self.itos[i] if i < len(self.itos) else b""
            if isinstance(t, bytes):
                try: parts.append(t.decode("utf-8"))
                except Exception: parts.append("�")
            else:
                parts.append(str(t))
        return "".join(parts)

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"merges": [[a.hex(), b.hex()] for a, b in self.merges]}, f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        tok = cls()
        tok.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in data["merges"]]
        tok._build()
        return tok

    @classmethod
    def load_any(cls, path):
        """Load from either the original JSON format or a pickle dump.

        Why this exists
        ---------------
        Phase-1 models were trained with different BPE tables.  Some exports
        are the original JSON (list of hex merges); others are full pickle
        dumps of the tokenizer object or of a dict containing "merges".
        This helper accepts both so you can drop .pkl files straight into
        DATASET_DIR without converting them first.
        """
        path = str(path)
        if path.endswith((".pkl", ".pickle")):
            import pickle
            with open(path, "rb") as f:
                obj = pickle.load(f)
            if isinstance(obj, cls):
                return obj
            if isinstance(obj, dict):
                tok = cls()
                if "merges" in obj:
                    merges = obj["merges"]
                    # hex-encoded (from .json export) or raw bytes
                    if merges and isinstance(merges[0][0], str):
                        tok.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in merges]
                    else:
                        tok.merges = [(bytes(a), bytes(b)) for a, b in merges]
                    tok._build()
                    return tok
                # full state-dict style
                for k, v in obj.items():
                    setattr(tok, k, v)
                if not getattr(tok, "stoi", None):
                    tok._build()
                return tok
            raise TypeError(f"Unrecognised pickle content in {path}: {type(obj)}")
        # original JSON path
        return cls.load(path)


assert os.path.exists(TOKENIZER_PATH), f"Tokenizer not found at {TOKENIZER_PATH}"
tokenizer = BPETokenizer.load_any(TOKENIZER_PATH)
print(f"Tokenizer loaded | vocab_size = {tokenizer.vocab_size} | from {TOKENIZER_PATH}")

# Pre-load every available tokenizer so Phase 4 can switch per experiment.
# Specials (pad/unk/bos/eos = 0/1/2/3) are identical across files; only merges/vocab differ.
TOKENIZERS = {}
for _name, _path in TOK_PATHS.items():
    if os.path.exists(_path):
        TOKENIZERS[_name] = BPETokenizer.load_any(_path)
        print(f"  tok[{_name}]: vocab={TOKENIZERS[_name].vocab_size}  <- {_path}")
    else:
        print(f"  tok[{_name}]: MISSING {_path}")
# keep global `tokenizer` pointing at the BACKBONE selected in the config cell
tokenizer = TOKENIZERS.get(BACKBONE, tokenizer)
print(f"Active tokenizer for BACKBONE={BACKBONE}: vocab={tokenizer.vocab_size}")


Tokenizer loaded | vocab_size = 30000 | from /kaggle/working/phase1_models/legal_bpe_mla_moe.json
  tok[custom_mha]: vocab=24000  <- /kaggle/working/phase1_models/legal_bpe_custom_mha.json
  tok[mla_moe]: vocab=30000  <- /kaggle/working/phase1_models/legal_bpe_mla_moe.json
  tok[gqa_conv]: vocab=30000  <- /kaggle/working/phase1_models/legal_bpe_gqa_conv.json
  tok[gqa_alibi]: vocab=32000  <- /kaggle/working/phase1_models/legal_bpe_tokenizer_gqa_alibi.pkl
Active tokenizer for BACKBONE=mla_moe: vocab=30000


In [ ]:
for name, tok in TOKENIZERS.items():
    if not hasattr(tok, 'unk_id'):
        tok.unk_id = 1
    if not hasattr(tok, 'pad_id'):
        tok.pad_id = 0
    if not hasattr(tok, 'bos_id'):
        tok.bos_id = 2
    if not hasattr(tok, 'eos_id'):
        tok.eos_id = 3

## 2. Shared building blocks (RMSNorm, RoPE)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.weight = self.w  # alias for alibi ckpts
        self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.w


class RoPE(nn.Module):
    def __init__(self, hd, max_len, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
        fr = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", fr.cos(), persistent=False)
        self.register_buffer("sin", fr.sin(), persistent=False)
    def forward(self, x, start=0):                      # x: (b, heads, t, hd)
        t = x.size(2)
        cos = self.cos[start:start + t].to(x.dtype)
        sin = self.sin[start:start + t].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((x1 * cos - x2 * sin, x2 * cos + x1 * sin), dim=-1)


## 3. Architecture A – LegalLM v2 (standard decoder)

In [ ]:
# Custom MHA

class AttentionV2(nn.Module):
    def __init__(self, dim, heads, block_size):
        super().__init__()
        assert dim % heads == 0
        self.h, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.rope = RoPE(self.hd, block_size)
    def forward(self, x, mask=None, cache=None, start=0):
        b, t, c = x.shape
        q, k, v = self.qkv(x).reshape(b, t, 3, self.h, self.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope(q, start), self.rope(k, start) # rotate q and key vectors
        if cache is not None: # KV caching for inference to be faster
            if cache[0] is not None:
                k = torch.cat([cache[0], k], 2)
                v = torch.cat([cache[1], v], 2)
            cache = (k, v)
        if mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        elif t > 1:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            y = F.scaled_dot_product_attention(q, k, v)
        return self.proj(y.transpose(1, 2).reshape(b, t, c)), cache


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=8/3):
        super().__init__()
        hid = int(mult * dim / 64) * 64
        self.w13 = nn.Linear(dim, 2 * hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
    def forward(self, x):
        a, b = self.w13(x).chunk(2, dim=-1)
        return self.w2(F.silu(a) * b) # Swiglu is basically W * (silu(a) * b) + bias


class BlockV2(nn.Module): # Block for MHA
    def __init__(self, dim, heads, block_size):
        super().__init__()
        self.n1, self.n2 = RMSNorm(dim), RMSNorm(dim)
        self.attn = AttentionV2(dim, heads, block_size)
        self.mlp = SwiGLU(dim)
    def forward(self, x, mask=None, cache=None, start=0):
        h, cache = self.attn(self.n1(x), mask, cache, start) # Pre-norm, fisrt normalise then add
        x = x + h
        return x + self.mlp(self.n2(x)), cache


class LegalLMv2(nn.Module):
    """Standard RoPE + RMSNorm + SwiGLU decoder (Phase-1 v2)."""
    def __init__(self, vocab_size, dim=768, layers=8, heads=12, block_size=1024):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([BlockV2(dim, heads, block_size) for _ in range(layers)])
        self.ln_f = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for blk in self.blocks:
            x, _ = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def step(self, idx, caches, start, mask=None):
        x = self.wte(idx)
        new = []
        for blk, c in zip(self.blocks, caches):
            x, c = blk(x, mask, c, start)
            new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new


def load_v2(path=None, device=DEVICE):
    path = path or CKPT_V2
    assert os.path.exists(path), f"v2 checkpoint missing: {path}"
    ck = torch.load(path, map_location="cpu", weights_only=False)
    # tolerate both raw state-dict and full training checkpoint
    if isinstance(ck, dict) and "model" in ck:
        state = ck["model"]
        cfg = ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        state = ck["model_state"]
        cfg = ck.get("config", {})
    else:
        state = ck
        cfg = {}
    # strip possible torch.compile prefix
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    vocab = cfg.get("vocab_size", state.get("wte.weight", torch.empty(0)).shape[0] or tokenizer.vocab_size)
    dim   = cfg.get("dim", cfg.get("n_embd", 768))
    layers= cfg.get("layers", cfg.get("n_layer", 8))
    heads = cfg.get("heads", cfg.get("n_head", 12))
    bs    = cfg.get("block_size", 1024)
    model = LegalLMv2(vocab, dim, layers, heads, bs)
    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    print(f"v2 loaded | params={sum(p.numel() for p in model.parameters())/1e6:.1f}M | from {path}")
    return model

## 4. Architecture B – LegalLM v3 (MLA + domain-guided MoE)

In [ ]:
# Default hyper-params used when the checkpoint does not embed a config
V3_DEFAULTS = dict(
    dim=640, layers=8, heads=10, block_size=1024,
    d_c=128, d_cq=256, d_nope=64, d_rope=32, d_v=64,
    n_routed=8, topk=2,
    expert_hid=512, shared_hid=1024, dense_hid=1728,
    dense_first=1, n_domains=4,
) # mLA MOE


class MLA(nn.Module):
    """Multi-head Latent Attention (DeepSeek-style) with decoupled RoPE."""
    def __init__(self, c):
        super().__init__()
        dim, H = c["dim"], c["heads"]
        self.h = H
        self.dc, self.dn, self.dr, self.dv = c["d_c"], c["d_nope"], c["d_rope"], c["d_v"]
        self.dq = nn.Linear(dim, c["d_cq"], bias=False)
        self.q_norm = RMSNorm(c["d_cq"])
        self.uq = nn.Linear(c["d_cq"], H * (self.dn + self.dr), bias=False)
        self.dkv = nn.Linear(dim, self.dc + self.dr, bias=False)
        self.kv_norm = RMSNorm(self.dc)
        self.uk = nn.Linear(self.dc, H * self.dn, bias=False)
        self.uv = nn.Linear(self.dc, H * self.dv, bias=False)
        self.o = nn.Linear(H * self.dv, dim, bias=False)
        self.rope = RoPE(self.dr, c["block_size"])
        self.scale = 1.0 / math.sqrt(self.dn + self.dr)

    def forward(self, x, mask=None, cache=None, start=0):
        b, t, _ = x.shape
        q = self.uq(self.q_norm(self.dq(x))).reshape(b, t, self.h, self.dn + self.dr).transpose(1, 2)
        q_nope, q_rope = q.split([self.dn, self.dr], dim=-1)
        ckv, k_rope = self.dkv(x).split([self.dc, self.dr], dim=-1)
        ckv = self.kv_norm(ckv)
        q_rope = self.rope(q_rope, start)
        k_rope = self.rope(k_rope.unsqueeze(1), start)  # (b,1,t,dr)
        if cache is not None:
            if cache[0] is not None:
                ckv = torch.cat([cache[0], ckv], 1)
                k_rope = torch.cat([cache[1], k_rope], 2)
            cache = (ckv, k_rope)
        k_nope = self.uk(ckv).reshape(b, -1, self.h, self.dn).transpose(1, 2)
        v = self.uv(ckv).reshape(b, -1, self.h, self.dv).transpose(1, 2)
        k = torch.cat([k_nope, k_rope.expand(-1, self.h, -1, -1)], dim=-1)
        q = torch.cat([q_nope, q_rope], dim=-1)
        if mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, scale=self.scale)
        elif t > 1:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, scale=self.scale)
        else:
            y = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
        return self.o(y.transpose(1, 2).reshape(b, t, -1)), cache


class Expert(nn.Module):
    def __init__(self, dim, hid):
        super().__init__()
        self.w1 = nn.Linear(dim, hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
        self.w3 = nn.Linear(dim, hid, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class DeepSeekMoE(nn.Module):
    def __init__(self, c):
        super().__init__()
        dim = c["dim"]
        self.n_routed, self.topk = c["n_routed"], c["topk"]
        self.experts = nn.ModuleList([Expert(dim, c["expert_hid"]) for _ in range(self.n_routed)])
        self.shared = Expert(dim, c["shared_hid"])
        self.gate = nn.Linear(dim, self.n_routed, bias=False)
        self.n_domains = c.get("n_domains", 4)

    def forward(self, x, domain_ids=None):
        # x: (b, t, d)
        b, t, d = x.shape
        logits = self.gate(x)                                          # (b,t,n)
        scores = F.softmax(logits, dim=-1)
        topv, topi = scores.topk(self.topk, dim=-1)                    # (b,t,k)
        topv = topv / topv.sum(-1, keepdim=True)
        out = self.shared(x)
        flat = x.reshape(-1, d)
        for k in range(self.topk):
            idx = topi[..., k].reshape(-1)                             # (b*t,)
            w = topv[..., k].reshape(-1, 1)
            for e in range(self.n_routed):
                mask = idx == e
                if mask.any():
                    out.view(-1, d)[mask] = out.view(-1, d)[mask] + (w[mask] * self.experts[e](flat[mask])).to(out.dtype)
        return out


class DenseFFN(nn.Module):
    def __init__(self, dim, hid):
        super().__init__()
        self.w13 = nn.Linear(dim, 2 * hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
    def forward(self, x):
        a, b = self.w13(x).chunk(2, dim=-1)
        return self.w2(F.silu(a) * b)


class BlockV3(nn.Module):
    def __init__(self, c, use_moe=True):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c["dim"]), RMSNorm(c["dim"])
        self.attn = MLA(c)
        self.ffn = DeepSeekMoE(c) if use_moe else DenseFFN(c["dim"], c["dense_hid"])
    def forward(self, x, mask=None, cache=None, start=0, domain_ids=None):
        h, cache = self.attn(self.n1(x), mask, cache, start)
        x = x + h
        if isinstance(self.ffn, DeepSeekMoE):
            x = x + self.ffn(self.n2(x), domain_ids)
        else:
            x = x + self.ffn(self.n2(x))
        return x, cache


class LegalLMv3(nn.Module):
    """MLA + domain-guided MoE decoder (Phase-1 v3)."""
    def __init__(self, vocab_size, c=None):
        super().__init__()
        c = {**V3_DEFAULTS, **(c or {})}
        self.cfg = c
        self.block_size = c["block_size"]
        self.wte = nn.Embedding(vocab_size, c["dim"])
        self.blocks = nn.ModuleList([
            BlockV3(c, use_moe=(i >= c["dense_first"])) for i in range(c["layers"])
        ])
        self.ln_f = RMSNorm(c["dim"])
        self.lm_head = nn.Linear(c["dim"], vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

    def forward(self, idx, targets=None, domain_ids=None):
        x = self.wte(idx)
        for blk in self.blocks:
            x, _ = blk(x, domain_ids=domain_ids)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def step(self, idx, caches, start, mask=None):
        x = self.wte(idx)
        new = []
        for blk, c in zip(self.blocks, caches):
            x, c = blk(x, mask, c, start)
            new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new


def load_v3(path=None, device=DEVICE):
    path = path or CKPT_V3
    assert os.path.exists(path), f"v3 checkpoint missing: {path}"
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, dict) and "model" in ck:
        state = ck["model"]
        cfg = ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        state = ck["model_state"]
        cfg = ck.get("config", {})
    else:
        state = ck
        cfg = {}
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    # merge defaults with any saved config
    c = {**V3_DEFAULTS, **cfg}
    if "wte.weight" in state:
        vocab = state["wte.weight"].shape[0]
        c["dim"] = state["wte.weight"].shape[1]
    else:
        vocab = tokenizer.vocab_size
    model = LegalLMv3(vocab, c)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print("v3 missing keys (first 5):", missing[:5])
    model.to(device).eval()
    print(f"v3 loaded | params={sum(p.numel() for p in model.parameters())/1e6:.1f}M | from {path}")
    return model

## 5. Architecture C – GQA + Dual-path FFN

In [ ]:
GQA_DEFAULTS = dict(
    dim=768, layers=8, heads=12, kv_heads=4, block_size=1024,
    qk_norm=True, ffn_mult=3, conv_k=4, dual_path=True,
    lambda_per_channel=True, lambda_init=0.0,
)


class GQAAttention(nn.Module):
    """Grouped-query attention + RoPE (+ optional QK-Norm)."""
    def __init__(self, c):
        super().__init__()
        d, self.h, self.hk = c["dim"], c["heads"], c["kv_heads"]
        assert self.h % self.hk == 0
        self.hd = d // self.h
        self.q = nn.Linear(d, self.h * self.hd, bias=False)
        self.kv = nn.Linear(d, 2 * self.hk * self.hd, bias=False)
        self.o = nn.Linear(self.h * self.hd, d, bias=False)
        self.qn = RMSNorm(self.hd) if c.get("qk_norm", True) else nn.Identity()
        self.kn = RMSNorm(self.hd) if c.get("qk_norm", True) else nn.Identity()
        self.rope = RoPE(self.hd, c["block_size"])

    def forward(self, x, mask=None, cache=None, start=0):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.h, self.hd).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.hk, self.hd).permute(2, 0, 3, 1, 4)
        q = self.rope(self.qn(q), start)
        k = self.rope(self.kn(k), start)
        if cache is not None:
            if cache[0] is not None:
                k = torch.cat([cache[0], k], 2)
                v = torch.cat([cache[1], v], 2)
            cache = (k, v)
        # expand KV heads to match Q heads
        if self.hk != self.h:
            rep = self.h // self.hk
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)
        if mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        elif t > 1:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            y = F.scaled_dot_product_attention(q, k, v)
        return self.o(y.transpose(1, 2).reshape(b, t, -1)), cache


class CausalDWConv(nn.Module):
    """Depth-wise causal 1-D convolution (kernel size K, left-padded)."""
    def __init__(self, dim, k=4):
        super().__init__()
        self.k = k
        self.conv = nn.Conv1d(dim, dim, k, groups=dim, bias=False)
    def forward(self, x):                                   # (b,t,d)
        x = x.transpose(1, 2)                               # (b,d,t)
        x = F.pad(x, (self.k - 1, 0))
        return self.conv(x).transpose(1, 2)


class DualPathFFN(nn.Module):
    """SwiGLU + λ * causal DWConv."""
    def __init__(self, c):
        super().__init__()
        dim = c["dim"]
        hid = int(c.get("ffn_mult", 3) * dim / 64) * 64
        self.w13 = nn.Linear(dim, 2 * hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
        self.dw = CausalDWConv(dim, c.get("conv_k", 4))
        if c.get("lambda_per_channel", True):
            self.lam = nn.Parameter(torch.full((dim,), c.get("lambda_init", 0.0)))
        else:
            self.lam = nn.Parameter(torch.tensor(c.get("lambda_init", 0.0)))

    def forward(self, x):
        a, b = self.w13(x).chunk(2, dim=-1)
        swiglu = self.w2(F.silu(a) * b)
        return swiglu + torch.sigmoid(self.lam) * self.dw(x)


class BlockGQA(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c["dim"]), RMSNorm(c["dim"])
        self.attn = GQAAttention(c)
        self.ffn = DualPathFFN(c) if c.get("dual_path", True) else SwiGLU(c["dim"], c.get("ffn_mult", 3))
    def forward(self, x, mask=None, cache=None, start=0):
        h, cache = self.attn(self.n1(x), mask, cache, start)
        x = x + h
        return x + self.ffn(self.n2(x)), cache


class DualPathLM(nn.Module):
    """GQA + Dual-path FFN decoder."""
    def __init__(self, vocab_size, c=None):
        super().__init__()
        c = {**GQA_DEFAULTS, **(c or {})}
        self.cfg = c
        self.block_size = c["block_size"]
        self.wte = nn.Embedding(vocab_size, c["dim"])
        self.blocks = nn.ModuleList([BlockGQA(c) for _ in range(c["layers"])])
        self.ln_f = RMSNorm(c["dim"])
        self.lm_head = nn.Linear(c["dim"], vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for blk in self.blocks:
            x, _ = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def step(self, idx, caches, start, mask=None):
        x = self.wte(idx)
        new = []
        for blk, c in zip(self.blocks, caches):
            x, c = blk(x, mask, c, start)
            new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new


def load_gqa(path=None, device=DEVICE):
    path = path or CKPT_GQA
    assert os.path.exists(path), f"GQA checkpoint missing: {path}"
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, dict) and "model" in ck:
        state = ck["model"]
        cfg = ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        state = ck["model_state"]
        cfg = ck.get("config", {})
    else:
        state = ck
        cfg = {}
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    c = {**GQA_DEFAULTS, **cfg}
    if "wte.weight" in state:
        vocab = state["wte.weight"].shape[0]
        c["dim"] = state["wte.weight"].shape[1]
    else:
        vocab = tokenizer.vocab_size
    model = DualPathLM(vocab, c)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print("GQA missing keys (first 5):", missing[:5])
    model.to(device).eval()
    print(f"GQA loaded | params={sum(p.numel() for p in model.parameters())/1e6:.1f}M | from {path}")
    return model

## 6. Architecture D – ALiBi + GQA


In [ ]:
def alibi_slopes(heads):
    n = 2 ** math.floor(math.log2(heads))
    s = [2.0 ** (-8.0 / n * (i + 1)) for i in range(n)]
    if n < heads:
        extra = 2.0 ** (-4.0 / n)
        s += [extra ** (2 * i + 1) for i in range(heads - n)]
    return torch.tensor(s, dtype=torch.float32)

def alibi_causal_bias(slopes, tq, tk, device):
    qpos = torch.arange(tk - tq, tk, device=device)[:, None]
    kpos = torch.arange(tk, device=device)[None, :]
    dist = (qpos - kpos).float()
    bias = -slopes.to(device).view(1, -1, 1, 1) * dist.clamp(min=0)
    return bias.masked_fill((dist < 0).view(1, 1, tq, tk), float("-inf"))

class ALiBiGQA(nn.Module):
    def __init__(self, dim, heads, kv_heads, dropout=0.0):
        super().__init__()
        assert dim % heads == 0 and heads % kv_heads == 0
        self.h, self.kvh, self.hd = heads, kv_heads, dim // heads
        self.rep = heads // kv_heads
        self.dropout = dropout
        self.q = nn.Linear(dim, dim, bias=False)
        self.kv = nn.Linear(dim, 2 * kv_heads * self.hd, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x, bias, past=None, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.h, self.hd).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.kvh, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        if self.rep > 1:
            k = k.repeat_interleave(self.rep, dim=1); v = v.repeat_interleave(self.rep, dim=1)
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=bias.to(q.dtype) if bias is not None else None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(b, t, -1)), present

class SwiGLU_A(nn.Module):
    def __init__(self, dim, dropout=0.0):
        super().__init__()
        hid = int(2 * 4 * dim / 3); hid = (hid + 7) // 8 * 8
        self.w1 = nn.Linear(dim, hid, bias=False)
        self.w3 = nn.Linear(dim, hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class BlockAlibi(nn.Module):
    def __init__(self, dim, heads, kv_heads, dropout=0.0):
        super().__init__()
        self.n1, self.n2 = RMSNorm(dim), RMSNorm(dim)
        self.attn = ALiBiGQA(dim, heads, kv_heads, dropout)
        self.mlp = SwiGLU_A(dim, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class LegalLMAlibi(nn.Module):
    def __init__(self, vocab_size, dim=768, layers=12, heads=12, kv_heads=6, block_size=1024, dropout=0.0):
        super().__init__()
        self.block_size, self.heads = block_size, heads
        self.kind = "gqa_alibi"
        self.wte = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([BlockAlibi(dim, heads, kv_heads, dropout) for _ in range(layers)])
        self.ln_f = RMSNorm(dim)  # training notebook used norm_f
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.register_buffer("slopes", alibi_slopes(heads), persistent=False)
    def forward(self, idx, targets=None, past=None, use_cache=False, key_pad=None):
        b, t = idx.shape
        past_len = 0 if past is None else past[0][0].shape[2]
        bias = alibi_causal_bias(self.slopes, t, past_len + t, idx.device)
        if key_pad is not None:
            bias = bias + key_pad.view(b, 1, 1, -1).float() * (-3e4)
        x = self.drop(self.wte(idx))
        presents = []
        for i, blk in enumerate(self.blocks):
            p = None if past is None else past[i]
            x, pr = blk(x, bias, p, use_cache)
            if use_cache: presents.append(pr)
        logits = self.lm_head(self.ln_f(x))
        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1)) if targets is not None else None
        if use_cache:
            return logits, loss, presents
        return logits, loss

def load_alibi(path=None, device=DEVICE):
    path = path or CKPT_ALIBI
    assert os.path.exists(path), f"alibi checkpoint missing: {path}"
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(ck, dict) and "model" in ck:
        state, cfg = ck["model"], dict(ck.get("config") or {})
    elif isinstance(ck, dict) and "model_state" in ck:
        state, cfg = ck["model_state"], dict(ck.get("config") or {})
    else:
        state, cfg = ck, {}
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    fixed = {}
    for k, v in state.items():
        if k.startswith("norm_f."):
            k = "ln_f." + k[len("norm_f."):]
        # our RMSNorm param is .w, checkpoints use .weight
        if k.endswith(".weight") and (
            k == "ln_f.weight"
            or ".n1.weight" in k
            or ".n2.weight" in k
            or ".q_norm.weight" in k
            or ".kv_norm.weight" in k
            or ".qn.weight" in k
            or ".kn.weight" in k
        ):
            k = k[: -len("weight")] + "w"
        fixed[k] = v
    state = fixed
    if "wte.weight" in state:
        vocab, dim = state["wte.weight"].shape
    else:
        vocab = int(cfg.get("vocab_size", tokenizer.vocab_size))
        dim = int(cfg.get("dim", cfg.get("n_embd", 768)))
    layers = int(cfg.get("layers", cfg.get("n_layer", 12)))
    heads = int(cfg.get("heads", cfg.get("n_head", 12)))
    kv = int(cfg.get("kv_heads", cfg.get("n_kv_head", 6)))
    bs = int(cfg.get("block_size", 1024))
    model = LegalLMAlibi(vocab, dim, layers, heads, kv, bs)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print("[alibi] missing (first 5):", missing[:5])
    model.to(device).eval()
    print(f"[alibi] {sum(p.numel() for p in model.parameters())/1e6:.1f}M | dim={dim} layers={layers} | {path}")
    return model


## 1. Load Phase-1 backbone(s)

Loads the selected `BACKBONE` (and optionally the others for comparison).


In [ ]:
MODELS = {}
_loaders = {"custom_mha": load_v2, "mla_moe": load_v3, "gqa_conv": load_gqa, "gqa_alibi": load_alibi}
_ckpts   = {"custom_mha": CKPT_V2, "mla_moe": CKPT_V3, "gqa_conv": CKPT_GQA, "gqa_alibi": CKPT_ALIBI}

for name in (BACKBONE,):
    if os.path.exists(_ckpts[name]):
        MODELS[name] = _loaders[name](_ckpts[name])
    else:
        print(f"WARNING: {name} checkpoint missing: {_ckpts[name]}")

assert BACKBONE in MODELS, f"Selected BACKBONE={BACKBONE} failed to load"
phase1_lm = MODELS[BACKBONE]
print(f"\nActive Phase-1 backbone: {BACKBONE} | {sum(p.numel() for p in phase1_lm.parameters())/1e6:.1f}M params")

# Safety: tokenizer vocab must match the embedding matrix of the loaded checkpoint
_model_vocab = phase1_lm.wte.weight.shape[0]
_tok_vocab   = tokenizer.vocab_size
print(f"vocab check  : model={_model_vocab}  tokenizer={_tok_vocab}")
assert _model_vocab == _tok_vocab, (
    f"Vocab mismatch! Checkpoint expects {_model_vocab} tokens but "
    f"tokenizer has {_tok_vocab}.  Point TOK_PATHS['{BACKBONE}'] at the correct file."
)


v3 missing keys (first 5): ['blocks.0.n1.weight', 'blocks.0.n2.weight', 'blocks.0.attn.q_norm.weight', 'blocks.0.attn.kv_norm.weight', 'blocks.1.n1.weight']
v3 loaded | params=100.1M | from /kaggle/working/phase1_models/legal_lm_mla_moe_1k.pt

Active Phase-1 backbone: mla_moe | 100.1M params
vocab check  : model=30000  tokenizer=30000


---
# Phase 7 – Prior Case Retrieval


In [ ]:
import glob
import pickle
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from huggingface_hub import hf_hub_download
from torch.utils.checkpoint import checkpoint
from torch.optim.lr_scheduler import LambdaLR

# True = tiny smoke test, False = full experiments
QUICK_RUN = False
TAG = "_quick" if QUICK_RUN else ""

# Output folders
P7_OUT = "/kaggle/working/phase7" if os.path.isdir("/kaggle/working") else "./phase7"
P7_DATA = os.path.join(P7_OUT, "data")
P7_CKPT = os.path.join(P7_OUT, "ckpt")
P7_SUBS = os.path.join(P7_OUT, "submissions")
P7_CACHE = os.path.join(P7_OUT, "cache")
for folder in (P7_DATA, P7_CKPT, P7_SUBS, P7_CACHE):
    os.makedirs(folder, exist_ok=True)

# Retrieval and tokenisation settings
TOP_K = 7
MAX_DOC_CHARS = 100_000
MAX_DOC_TOKENS = 16_000
CHUNK_LEN = 512
INFER_CHUNKS = 2 if QUICK_RUN else 8       # chunks per document at inference
DEV_EVAL_CHUNKS = 2 if QUICK_RUN else 4    # chunks per document for per-epoch dev checks
TRAIN_CHUNKS = 2                           # random chunks per document in training
POOLING = "mean"                           # "mean" | "last" | "mean_last"

# Training settings
BATCH_Q = 8
TEMP = 0.05
LOSS_KIND = "infonce"                      # "infonce" | "triplet"
MARGIN = 0.2
NEG_MODE = "bm25"                          # "bm25" | "random"
HN_DEPTH = 40
HN_SKIP = 2                                # skip the top BM25 hits (may be unlabeled positives)
USE_GRAD_CKPT = True
TRAIN_LIMIT = 64 if QUICK_RUN else None

METHOD_CFG = {
    "full":    {"lr": 2e-5, "epochs": 10},
    "partial": {"lr": 5e-5, "epochs": 1},
    "lorma":   {"lr": 1e-3, "epochs": 5},
    "ia3":     {"lr": 3e-3, "epochs": 1},
    "kd":      {"lr": 5e-5, "epochs": 1},
}
if QUICK_RUN:
    for cfg in METHOD_CFG.values():
        cfg["epochs"] = 1

LORMA_RANK = 16
LORMA_ALPHA = 32
PARTIAL_LAYERS = 4
IA3_TARGETS = ("qkv", "kv", "uk", "uv", "w13", "w1", "w3", "w2")
KD_KEEP_RATIO = 0.5    # student keeps this share of the layers
KD_ALPHA = 0.5         # weight of the soft (teacher) loss
KD_BETA = 1.0          # weight of the embedding alignment loss

assert CHUNK_LEN <= phase1_lm.block_size, "CHUNK_LEN exceeds the model block size"

# Free GPU memory, every experiment reloads fresh Phase-1 weights
phase1_lm.to("cpu")


def tagged(exp, variant=""):
    """Unique name for an experiment, used in all file names."""
    return f"{BACKBONE}__{exp}{variant}{TAG}"


# Results survive restarts
RESULTS_PATH = os.path.join(P7_OUT, "results.json")
RESULTS = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        RESULTS = json.load(f)

print("quick run:", QUICK_RUN, "| backbone:", BACKBONE, "| out:", P7_OUT)

quick run: False | backbone: mla_moe | out: /kaggle/working/phase7


In [ ]:
SPLIT_FILES = {
    "train": ("Phase_7_PCR/Train/train.json", "Phase_7_PCR/Train/train_candidates.json"),
    "dev":   ("Phase_7_PCR/Dev/dev.json", "Phase_7_PCR/Dev/dev_candidates.json"),
    "test":  ("Phase_7_PCR/Test/test.json", "Phase_7_PCR/Test/test_candidates.json"),
}
ID_KEYS = ("id", "query_id", "case_id", "doc_id", "qid", "cid", "candidate_id", "_id")


def fetch(rel_path):
    """Use the local copy if present, otherwise download from the HF dataset."""
    local = os.path.join(P7_DATA, rel_path)
    if os.path.exists(local):
        return local
    return hf_hub_download(repo_id=HF_REPO, filename=rel_path, repo_type="dataset",
                           token=HF_TOKEN, local_dir=P7_DATA)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def to_records(raw):
    """Return {case_id: record} from either a dict file or a list file."""
    if isinstance(raw, dict):
        return {str(k): v for k, v in raw.items()}
    records = {}
    for item in raw:
        key = next((item[k] for k in ID_KEYS if k in item), None)
        assert key is not None, f"no id key found in record keys: {list(item.keys())}"
        records[str(key)] = item
    return records


def get_text(rec):
    """Case text as one string (files store a list of sentences)."""
    if isinstance(rec, dict):
        rec = rec.get("text", "")
    if isinstance(rec, list):
        rec = " ".join(str(s) for s in rec)
    return str(rec)


def load_split(split):
    q_file, c_file = SPLIT_FILES[split]
    queries = to_records(load_json(fetch(q_file)))
    cands = to_records(load_json(fetch(c_file)))
    q_ids = list(queries.keys())
    c_ids = list(cands.keys())
    gold = {}
    for qid in q_ids:
        rec = queries[qid]
        rel = rec.get("relevant_candidates", []) if isinstance(rec, dict) else []
        gold[qid] = set(str(x) for x in rel)
    # A query must never retrieve itself if it also sits in the pool
    c_index = {cid: i for i, cid in enumerate(c_ids)}
    rows = [qi for qi, qid in enumerate(q_ids) if qid in c_index]
    cols = [c_index[q_ids[qi]] for qi in rows]
    return {
        "q_ids": q_ids,
        "c_ids": c_ids,
        "q_text": [get_text(queries[q]) for q in q_ids],
        "c_text": [get_text(cands[c]) for c in c_ids],
        "gold": gold,
        "self_pairs": (np.array(rows, dtype=int), np.array(cols, dtype=int)),
    }


DATA = {}
for split in ("train", "dev", "test"):
    DATA[split] = load_split(split)
    d = DATA[split]
    n_gold = [len(g) for g in d["gold"].values()]
    in_pool = [len(g & set(d["c_ids"])) / max(len(g), 1) for g in d["gold"].values()]
    avg_words = np.mean([len(t.split()) for t in d["c_text"][:500]])
    print(f"{split:5s} | queries={len(d['q_ids'])} | pool={len(d['c_ids'])} "
          f"| avg gold={np.mean(n_gold):.2f} | gold in pool={np.mean(in_pool):.2%} "
          f"| avg cand words (first 500)={avg_words:.0f} | query-in-pool={len(d['self_pairs'][0])}")

train | queries=827 | pool=4320 | avg gold=6.42 | gold in pool=98.79% | avg cand words (first 500)=6057 | query-in-pool=827
dev   | queries=118 | pool=1023 | avg gold=8.25 | gold in pool=100.00% | avg cand words (first 500)=8339 | query-in-pool=118
test  | queries=237 | pool=1727 | avg gold=0.00 | gold in pool=0.00% | avg cand words (first 500)=7223 | query-in-pool=237


In [ ]:
def micro_prf(ranked, gold, k=TOP_K):
    """Micro precision, recall and F1 over the top-k of every query."""
    tp = 0
    n_pred = 0
    n_gold = 0
    for qid, ids in ranked.items():
        top = ids[:k]
        tp += len(set(top) & gold[qid])
        n_pred += len(top)
        n_gold += len(gold[qid])
    precision = tp / max(n_pred, 1)
    recall = tp / max(n_gold, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1


def mask_self(scores, split):
    """Push the query's own document to the bottom if it is in the pool."""
    rows, cols = DATA[split]["self_pairs"]
    scores = scores.copy()
    scores[rows, cols] = -1e9
    return scores


def rank_ids(scores, split, k=TOP_K):
    """Top-k candidate ids per query, best first."""
    d = DATA[split]
    scores = mask_self(scores, split)
    top = np.argsort(-scores, axis=1)[:, :k]
    return {qid: [d["c_ids"][j] for j in top[i]] for i, qid in enumerate(d["q_ids"])}


def evaluate(scores, split):
    return micro_prf(rank_ids(scores, split), DATA[split]["gold"])


def recall_at(scores, split, k):
    """Share of gold documents found in the top-k (diagnostic only)."""
    return micro_prf(rank_ids(scores, split, k), DATA[split]["gold"], k)[1]


def score_path(name, split):
    return os.path.join(P7_CACHE, f"scores__{name}__{split}.npy")


def save_scores(name, split, scores):
    np.save(score_path(name, split), scores.astype(np.float32))


def load_scores(name, split):
    return np.load(score_path(name, split))


def has_scores(name, split):
    return os.path.exists(score_path(name, split))


def zscore(m):
    """Per-query standardisation so different scorers can be mixed."""
    return (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-8)


def write_submission(name, test_scores):
    """Write prediction.json, validate it, and zip it for upload."""
    d = DATA["test"]
    ranked = rank_ids(test_scores, "test")
    pool = set(d["c_ids"])
    for qid in d["q_ids"]:
        ids = ranked[qid]
        assert len(ids) == TOP_K and len(set(ids)) == TOP_K, f"bad list for {qid}"
        assert all(c in pool for c in ids), f"id outside the test pool for {qid}"
    folder = os.path.join(P7_SUBS, name)
    os.makedirs(folder, exist_ok=True)
    pred_path = os.path.join(folder, "prediction.json")
    with open(pred_path, "w") as f:
        json.dump(ranked, f, indent=2)
    zip_path = os.path.join(P7_SUBS, f"{name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, "prediction.json")
    print("submission written:", zip_path)
    return zip_path

In [ ]:
TOKEN_RE = r"(?u)\b[a-z][a-z]+\b"
_BM25_STATE = {}


def bm25_prepare(split):
    """Count terms once per split so k1 and b can be tuned cheaply."""
    d = DATA[split]
    vec = CountVectorizer(lowercase=True, token_pattern=TOKEN_RE, min_df=2,
                          max_df=0.6, dtype=np.float32)
    c_tf = vec.fit_transform(d["c_text"]).tocsr()
    q_tf = vec.transform(d["q_text"]).tocsr()
    doc_len = np.asarray(c_tf.sum(1)).ravel()
    df = np.asarray((c_tf > 0).sum(0)).ravel()
    n_docs = c_tf.shape[0]
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    _BM25_STATE[split] = (c_tf.tocoo(), q_tf, doc_len, idf)


def bm25_scores(split, k1=1.5, b=0.75):
    if split not in _BM25_STATE:
        bm25_prepare(split)
    coo, q_tf, doc_len, idf = _BM25_STATE[split]
    length_norm = k1 * (1 - b + b * doc_len[coo.row] / doc_len.mean())
    weights = coo.data * (k1 + 1) / (coo.data + length_norm) * idf[coo.col]
    W = sp.csr_matrix((weights, (coo.row, coo.col)), shape=coo.shape)
    q = q_tf.copy()
    q.data = np.log1p(q.data)      # long queries, so damp repeated terms
    return (q @ W.T).toarray().astype(np.float32)


def tfidf_scores(split):
    d = DATA[split]
    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_RE, ngram_range=(1, 2),
                          min_df=3, max_df=0.6, sublinear_tf=True,
                          max_features=300_000, dtype=np.float32)
    C = vec.fit_transform(d["c_text"])
    Q = vec.transform(d["q_text"])
    return (Q @ C.T).toarray().astype(np.float32)


STATUTE_RE = re.compile(r"\b(section|article|order|rule)s?\s+(\d+[a-z]?)\b")


def statute_analyzer(text):
    """Turn 'Section 302' or 'Article 21' mentions into tokens like section_302."""
    return [f"{kind}_{num}" for kind, num in STATUTE_RE.findall(text.lower())]


def statute_scores(split):
    d = DATA[split]
    vec = TfidfVectorizer(analyzer=statute_analyzer, sublinear_tf=True,
                          min_df=2, dtype=np.float32)
    C = vec.fit_transform(d["c_text"])
    Q = vec.transform(d["q_text"])
    return (Q @ C.T).toarray().astype(np.float32)


# Tune BM25 on dev with a tiny grid
best = (-1.0, 1.5, 0.75)
for k1 in (0.9, 1.5, 2.0):
    for b in (0.5, 0.75, 0.9):
        f1 = evaluate(bm25_scores("dev", k1, b), "dev")[2]
        if f1 > best[0]:
            best = (f1, k1, b)
BM25_K1, BM25_B = best[1], best[2]
print(f"best BM25 on dev: k1={BM25_K1} b={BM25_B} F1@7={best[0]:.4f}")

# Compute and cache every lexical scorer for every split
LEXICAL_FNS = {
    "bm25": lambda s: bm25_scores(s, BM25_K1, BM25_B),
    "tfidf": tfidf_scores,
    "statute": statute_scores,
}
for name, fn in LEXICAL_FNS.items():
    for split in ("train", "dev", "test"):
        if not has_scores(name, split):
            save_scores(name, split, fn(split))

rows = []
for name in LEXICAL_FNS:
    for split in ("train", "dev"):
        p, r, f = evaluate(load_scores(name, split), split)
        rows.append({"scorer": name, "split": split, "P@7": p, "R@7": r, "F1@7": f})
print(pd.DataFrame(rows).round(4).to_string(index=False))

best BM25 on dev: k1=2.0 b=0.9 F1@7=0.2390
 scorer split    P@7    R@7   F1@7
   bm25 train 0.1895 0.2067 0.1977
   bm25   dev 0.2603 0.2210 0.2390
  tfidf train 0.2902 0.3165 0.3028
  tfidf   dev 0.4165 0.3535 0.3824
statute train 0.1176 0.1283 0.1227
statute   dev 0.1804 0.1531 0.1656


In [ ]:
def tokenize_texts(texts, desc):
    docs = []
    for text in tqdm(texts, desc=desc, leave=False):
        ids = tokenizer.encode(clean_text(text[:MAX_DOC_CHARS]))
        docs.append(np.asarray(ids[:MAX_DOC_TOKENS], dtype=np.int32))
    return docs


def get_tokens(split):
    """Tokenise a split (queries + pool), cached per backbone tokenizer."""
    path = os.path.join(P7_CACHE, f"tok__{BACKBONE}__{split}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    out = {
        "q": tokenize_texts(DATA[split]["q_text"], f"{split} queries"),
        "c": tokenize_texts(DATA[split]["c_text"], f"{split} pool"),
    }
    with open(path, "wb") as f:
        pickle.dump(out, f)
    return out


TOK = {split: get_tokens(split) for split in ("train", "dev", "test")}
for split in TOK:
    lens = [len(x) for x in TOK[split]["c"]]
    print(f"{split}: pool docs={len(lens)} | median tokens={int(np.median(lens))} "
          f"| 90th pct={int(np.percentile(lens, 90))}")


def make_chunks(ids, n_chunks, train=False):
    """Cut a token array into windows of CHUNK_LEN-1 tokens, each starting with BOS.

    train=True  -> n_chunks random windows
    train=False -> cover the doc if short, else evenly spaced windows
    """
    body = CHUNK_LEN - 1
    n = len(ids)
    last_start = max(0, n - body)
    if train:
        starts = [random.randint(0, last_start) for _ in range(n_chunks)]
    elif n <= body * n_chunks:
        starts = list(range(0, max(n, 1), body))
    else:
        starts = np.linspace(0, last_start, n_chunks).astype(int).tolist()
    chunks = []
    for s in starts:
        piece = np.concatenate([[tokenizer.bos_id], ids[s:s + body]])
        chunks.append(piece.astype(np.int64))
    return chunks


def pad_batch(chunks):
    """Right-pad chunks. Causal attention means pads never affect real tokens."""
    lengths = torch.tensor([len(c) for c in chunks])
    ids = torch.full((len(chunks), int(lengths.max())), tokenizer.pad_id, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
    return ids, lengths

train: pool docs=4320 | median tokens=4589 | 90th pct=15593
dev: pool docs=1023 | median tokens=5094 | 90th pct=16000
test: pool docs=1727 | median tokens=4935 | 90th pct=16000


In [ ]:
def run_block(blk, x, bias, use_ckpt):
    """Run one backbone block, optionally with gradient checkpointing."""
    def fn(inp):
        out = blk(inp) if bias is None else blk(inp, bias)
        return out[0]
    if use_ckpt and torch.is_grad_enabled():
        return checkpoint(fn, x, use_reentrant=False)
    return fn(x)


def encode_hidden(model, ids, use_ckpt=False):
    """Phase-1 backbone up to the final norm (no LM head). Returns [B, T, D]."""
    x = model.wte(ids)
    bias = None
    if getattr(model, "kind", "") == "gqa_alibi":
        bias = alibi_causal_bias(model.slopes, ids.size(1), ids.size(1), ids.device)
    for blk in model.blocks:
        x = run_block(blk, x, bias, use_ckpt)
    return model.ln_f(x)


def pool(h, lengths, mode):
    """Turn token states [B, T, D] into one vector per chunk."""
    steps = torch.arange(h.size(1), device=h.device)[None, :]
    mask = (steps < lengths[:, None]).unsqueeze(-1).to(h.dtype)
    mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
    last = h[torch.arange(h.size(0), device=h.device), lengths - 1]
    if mode == "mean":
        return mean
    if mode == "last":
        return last
    if mode == "mean_last":
        return F.normalize(mean, dim=-1) + F.normalize(last, dim=-1)
    raise ValueError(mode)


def forward_chunks(model, chunks, modes, use_ckpt=False):
    """Embed a list of chunks. Returns {mode: [n_chunks, D]} in float32."""
    ids, lengths = pad_batch(chunks)
    ids = ids.to(DEVICE)
    lengths = lengths.to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        h = encode_hidden(model, ids, use_ckpt)
    h = h.float()
    return {m: pool(h, lengths, m) for m in modes}


@torch.inference_mode()
def encode_corpus(model, docs, n_chunks, modes=("mean",), batch_size=32):
    """Embed every doc as up to n_chunks chunks.

    Returns {mode: (E [N, n_chunks, D], counts [N])} on CPU.
    """
    model.eval()
    flat = []
    owner = []
    slot = []
    for di, ids in enumerate(docs):
        for si, chunk in enumerate(make_chunks(ids, n_chunks, train=False)):
            flat.append(chunk)
            owner.append(di)
            slot.append(si)
    order = np.argsort([len(c) for c in flat])      # similar lengths -> less padding
    dim = model.wte.weight.shape[1]
    out = {m: torch.zeros(len(flat), dim) for m in modes}
    for start in tqdm(range(0, len(order), batch_size), leave=False, desc="encode"):
        idx = order[start:start + batch_size]
        pooled = forward_chunks(model, [flat[i] for i in idx], modes)
        for m in modes:
            out[m][idx] = pooled[m].cpu()
    owner = torch.tensor(owner)
    slot = torch.tensor(slot)
    counts = torch.bincount(owner, minlength=len(docs))
    result = {}
    for m in modes:
        E = torch.zeros(len(docs), n_chunks, dim)
        E[owner, slot] = out[m]
        result[m] = (E, counts)
    return result


def doc_vectors(E, counts):
    """Mean of L2-normalised chunk vectors, normalised again. Returns [N, D]."""
    mask = (torch.arange(E.size(1))[None, :] < counts[:, None]).unsqueeze(-1).float()
    v = (F.normalize(E, dim=-1) * mask).sum(1) / mask.sum(1).clamp(min=1)
    return F.normalize(v, dim=-1)


def mean_scores(Eq, nq, Ec, nc):
    """Document-level cosine similarity of mean-pooled chunk vectors."""
    return (doc_vectors(Eq, nq) @ doc_vectors(Ec, nc).T).numpy()


def maxsim_scores(Eq, nq, Ec, nc):
    """For each query chunk take its best candidate chunk, then average over query chunks."""
    Eq = F.normalize(Eq, dim=-1)
    Ec = F.normalize(Ec, dim=-1).to(DEVICE)
    valid = (torch.arange(Ec.size(1))[None, :] < nc[:, None]).to(DEVICE)    # [N, C]
    out = torch.zeros(Eq.size(0), Ec.size(0))
    for i in range(Eq.size(0)):
        q = Eq[i, :int(nq[i])].to(DEVICE)                                   # [cq, D]
        sim = torch.einsum("qd,ncd->qnc", q, Ec)
        sim = sim.masked_fill(~valid[None], -1.0)
        out[i] = sim.max(-1).values.mean(0).cpu()
    return out.numpy()


def encode_split(model, split, n_chunks=None):
    """Encode queries and pool of one split. Returns (Eq, nq, Ec, nc)."""
    n_chunks = n_chunks or INFER_CHUNKS
    q = encode_corpus(model, TOK[split]["q"], n_chunks, (POOLING,))[POOLING]
    c = encode_corpus(model, TOK[split]["c"], n_chunks, (POOLING,))[POOLING]
    return q[0], q[1], c[0], c[1]

In [ ]:
class LoRMALinear(nn.Module):
    """Low-Rank Multiplicative Adaptation: W' = (I + s * B @ A) @ W.

    The frozen layer runs first, then a low-rank map is applied to its output.
    B starts at zero, so training begins exactly at the Phase-1 model.
    """
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        out_features = base.out_features
        self.A = nn.Parameter(torch.empty(rank, out_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / rank

    def forward(self, x):
        y = self.base(x)
        delta = F.linear(F.linear(y, self.A), self.B)
        return y + self.scale * delta


class IA3Linear(nn.Module):
    """IA3: learn one rescaling vector per output channel of a frozen layer."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        y = self.base(x)
        return y * self.scale.to(y.dtype)


def wrap_linears(blocks, factory, only=None, skip=("experts.", "gate")):
    """Replace nn.Linear layers inside the blocks with adapter wrappers."""
    count = 0
    for name, module in list(blocks.named_modules()):
        for child_name, child in list(module.named_children()):
            full = f"{name}.{child_name}" if name else child_name
            if not isinstance(child, nn.Linear):
                continue
            if any(s in full for s in skip):      # leave MoE experts and router alone
                continue
            if only is not None and child_name not in only:
                continue
            setattr(module, child_name, factory(child))
            count += 1
    return count


def make_student(model):
    """Smaller student: same width, evenly spaced subset of the Phase-1 layers."""
    n_layers = len(model.blocks)
    keep = max(2, int(round(n_layers * KD_KEEP_RATIO)))
    pick = np.linspace(0, n_layers - 1, keep).round().astype(int).tolist()
    model.blocks = nn.ModuleList([model.blocks[i] for i in pick])
    return model


def apply_method(model, method):
    """Freeze or unfreeze parameters and insert adapters for the chosen method."""
    for p in model.parameters():
        p.requires_grad = False
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
    elif method == "partial":
        for blk in model.blocks[-PARTIAL_LAYERS:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in model.ln_f.parameters():
            p.requires_grad = True
    elif method == "lorma":
        n = wrap_linears(model.blocks, lambda lin: LoRMALinear(lin, LORMA_RANK, LORMA_ALPHA))
        print("LoRMA layers:", n)
    elif method == "ia3":
        n = wrap_linears(model.blocks, IA3Linear, only=IA3_TARGETS)
        print("IA3 layers:", n)
    elif method != "zeroshot":
        raise ValueError(method)
    return model


def prepare_model(method):
    """Always start from the Phase-1 checkpoint."""
    model = _loaders[BACKBONE](_ckpts[BACKBONE])
    if method == "kd":
        model = make_student(model)
    model = apply_method(model, method)
    return model.to(DEVICE)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
def build_train_items():
    """One item per train query: gold indices plus BM25 hard negatives."""
    d = DATA["train"]
    c_index = {cid: i for i, cid in enumerate(d["c_ids"])}
    bm25_train = load_scores("bm25", "train")
    items = []
    for qi, qid in enumerate(d["q_ids"]):
        gold_idx = sorted(c_index[g] for g in d["gold"][qid] if g in c_index)
        if not gold_idx:
            continue
        gold_set = set(gold_idx)
        order = np.argsort(-bm25_train[qi])[:HN_DEPTH]
        hard = [int(j) for j in order if j not in gold_set and d["c_ids"][j] != qid]
        items.append({"q": qi, "gold": gold_idx, "hard": hard[HN_SKIP:]})
    if TRAIN_LIMIT:
        items = items[:TRAIN_LIMIT]
    print("train queries used:", len(items))
    return items


TRAIN_ITEMS = build_train_items()
N_TRAIN_POOL = len(DATA["train"]["c_ids"])


def train_embed(model, docs):
    """Embed docs with TRAIN_CHUNKS random windows each, mean-pool, normalise."""
    chunks = []
    for ids in docs:
        chunks.extend(make_chunks(ids, TRAIN_CHUNKS, train=True))
    pooled = forward_chunks(model, chunks, (POOLING,), use_ckpt=USE_GRAD_CKPT)[POOLING]
    pooled = F.normalize(pooled, dim=-1).view(len(docs), TRAIN_CHUNKS, -1)
    return F.normalize(pooled.mean(1), dim=-1)


def false_negative_mask(cand_idx, gold_sets):
    """True where a doc is gold for query i but is not query i's own positive."""
    mask = torch.zeros(len(gold_sets), len(cand_idx), dtype=torch.bool)
    for i, gold in enumerate(gold_sets):
        for j, c in enumerate(cand_idx):
            if c in gold and j != i:
                mask[i, j] = True
    return mask.to(DEVICE)


def step_loss(model, batch, teacher=None):
    """Contrastive loss with in-batch + hard negatives (optionally distilled)."""
    tr = TOK["train"]
    pos_idx = [random.choice(it["gold"]) for it in batch]
    neg_idx = []
    for it in batch:
        if NEG_MODE == "bm25" and it["hard"]:
            neg_idx.append(random.choice(it["hard"]))
        else:
            neg_idx.append(random.randrange(N_TRAIN_POOL))
    cand_idx = pos_idx + neg_idx            # positives first, so query i matches column i
    gold_sets = [set(it["gold"]) for it in batch]

    q = train_embed(model, [tr["q"][it["q"]] for it in batch])
    d = train_embed(model, [tr["c"][j] for j in cand_idx])
    sims = q @ d.T
    fn_mask = false_negative_mask(cand_idx, gold_sets)
    logits = (sims / TEMP).masked_fill(fn_mask, -1e4)
    target = torch.arange(len(batch), device=DEVICE)

    if LOSS_KIND == "infonce":
        loss = F.cross_entropy(logits, target)
    else:
        pos = sims[target, target].unsqueeze(1)
        valid = ~fn_mask
        valid[target, target] = False
        loss = (F.relu(MARGIN - pos + sims) * valid).sum() / valid.sum().clamp(min=1)

    if teacher is not None:
        tq = teacher["q"][[it["q"] for it in batch]].to(DEVICE)
        td = teacher["c"][cand_idx].to(DEVICE)
        t_logits = (tq @ td.T / TEMP).masked_fill(fn_mask, -1e4)
        soft = F.kl_div(F.log_softmax(logits, -1), F.softmax(t_logits, -1), reduction="batchmean")
        align = (1 - (q * tq).sum(-1)).mean() + (1 - (d * td).sum(-1)).mean()
        loss = (1 - KD_ALPHA) * loss + KD_ALPHA * soft + KD_BETA * align
    return loss


def ckpt_path(exp, kind):
    return os.path.join(P7_CKPT, f"{tagged(exp)}__{kind}.pt")


def trainable_state(model, names):
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}


def dev_f1_quick(model):
    """Cheap dev check used for model selection after every epoch."""
    Eq, nq, Ec, nc = encode_split(model, "dev", DEV_EVAL_CHUNKS)
    return evaluate(mean_scores(Eq, nq, Ec, nc), "dev")[2]


def train_model(model, exp, cfg, teacher=None):
    """Train, checkpoint every epoch (resumable), keep the best dev state."""
    items = list(TRAIN_ITEMS)
    params = [p for p in model.parameters() if p.requires_grad]
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.01)
    steps_per_epoch = math.ceil(len(items) / BATCH_Q)
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup = max(1, int(0.1 * total_steps))

    def lr_lambda(step):
        warm = min(1.0, (step + 1) / warmup)
        cosine = 0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps)))
        return warm * cosine

    sched = LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and AMP_DTYPE == torch.float16))

    start_epoch = 0
    best_f1 = -1.0
    history = []
    last_path = ckpt_path(exp, "last")
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["state"], strict=False)
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        start_epoch, best_f1, history = ck["epoch"], ck["best_f1"], ck["history"]
        print(f"resumed {exp} from epoch {start_epoch}")

    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        random.shuffle(items)
        losses = []
        for step in tqdm(range(steps_per_epoch), desc=f"{exp} epoch {epoch + 1}", leave=False):
            batch = items[step * BATCH_Q:(step + 1) * BATCH_Q]
            if len(batch) < 2:
                continue
            loss = step_loss(model, batch, teacher)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())

        f1 = dev_f1_quick(model)
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "dev_f1": f1})
        print(f"{exp} | epoch {epoch + 1} | loss {np.mean(losses):.4f} | dev F1@7 {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            torch.save({"state": trainable_state(model, names)}, ckpt_path(exp, "best"))
        torch.save({"state": trainable_state(model, names), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "epoch": epoch + 1,
                    "best_f1": best_f1, "history": history}, last_path)

    best = torch.load(ckpt_path(exp, "best"), map_location="cpu", weights_only=False)
    model.load_state_dict(best["state"], strict=False)
    return history

train queries used: 817


In [ ]:
def save_results():
    with open(RESULTS_PATH, "w") as f:
        json.dump(RESULTS, f, indent=2)


def finalize(model, exp, method, total, trainable, history):
    """Score dev and test with both similarity modes, save scores and the submission."""
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.time()
    dev_enc = encode_split(model, "dev")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    enc_secs = time.time() - t0
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    test_enc = encode_split(model, "test")

    scored = {}
    for variant, fn in (("", mean_scores), ("-maxsim", maxsim_scores)):
        dev_s = fn(*dev_enc)
        test_s = fn(*test_enc)
        save_scores(tagged(exp, variant), "dev", dev_s)
        save_scores(tagged(exp, variant), "test", test_s)
        scored[variant] = (evaluate(dev_s, "dev"), test_s)

    best_variant = max(scored, key=lambda v: scored[v][0][2])
    (p, r, f1), best_test = scored[best_variant]
    write_submission(tagged(exp, best_variant), best_test)

    row = {
        "name": tagged(exp), "exp": exp, "method": method,
        "total_params_M": round(total / 1e6, 2), "trainable_M": round(trainable / 1e6, 3),
        "dev_p": p, "dev_r": r, "dev_f1": f1,
        "dev_f1_mean": scored[""][0][2], "dev_f1_maxsim": scored["-maxsim"][0][2],
        "best_variant": best_variant or "mean",
        "dev_encode_secs": round(enc_secs, 1), "peak_gb": round(peak_gb, 2),
        "history": history,
    }
    RESULTS[tagged(exp)] = row
    save_results()
    print(f"{exp}: dev P={p:.4f} R={r:.4f} F1@7={f1:.4f} ({row['best_variant']})")
    return row


def run_experiment(exp, method, teacher=None, force=False):
    name = tagged(exp)
    if name in RESULTS and has_scores(name, "test") and not force:
        print(f"skip {exp}: already done (F1@7={RESULTS[name]['dev_f1']:.4f})")
        return RESULTS[name]
    model = prepare_model(method)
    total, trainable = count_params(model)
    print(f"{exp} | method={method} | total={total / 1e6:.1f}M | trainable={trainable / 1e6:.2f}M")
    history = []
    if method != "zeroshot":
        history = train_model(model, exp, METHOD_CFG[method], teacher)
    row = finalize(model, exp, method, total, trainable, history)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return row

In [ ]:
# # Pooling ablation on the untouched Phase-1 model (dev only)
# model = prepare_model("zeroshot")
# q_enc = encode_corpus(model, TOK["dev"]["q"], DEV_EVAL_CHUNKS, ("mean", "last", "mean_last"))
# c_enc = encode_corpus(model, TOK["dev"]["c"], DEV_EVAL_CHUNKS, ("mean", "last", "mean_last"))
# rows = []
# for mode in ("mean", "last", "mean_last"):
#     scores = mean_scores(*q_enc[mode], *c_enc[mode])
#     p, r, f1 = evaluate(scores, "dev")
#     rows.append({"pooling": mode, "P@7": p, "R@7": r, "F1@7": f1})
# print(pd.DataFrame(rows).round(4).to_string(index=False))
# del model, q_enc, c_enc
# gc.collect()
# torch.cuda.empty_cache()

# # Zero-shot dense baseline (mean pooling vs maxsim is stored for both)
# run_experiment("zeroshot", "zeroshot")

In [ ]:
# Required: LoRMA + at least two other techniques. Full FT is an extra.
for exp, method in (
    ("full", "full"),
    ("lorma", "lorma"),
):
    run_experiment(exp, method)

# Optional ablations (each needs a new exp name):
# NEG_MODE = "random"
# run_experiment("lorma_randneg", "lorma")
# NEG_MODE = "bm25"
# LOSS_KIND = "triplet"
# run_experiment("lorma_triplet", "lorma")
# LOSS_KIND = "infonce"

skip full: already done (F1@7=0.2857)
skip lorma: already done (F1@7=0.2702)


In [ ]:
def load_trained(exp, method):
    """Rebuild a fine-tuned model from Phase-1 weights plus its best saved state."""
    model = prepare_model(method)
    ck = torch.load(ckpt_path(exp, "best"), map_location="cpu", weights_only=False)
    model.load_state_dict(ck["state"], strict=False)
    return model.to(DEVICE).eval()


def build_teacher_tensors(exp, method):
    """Teacher embeddings for the whole train split, cached on disk."""
    path = os.path.join(P7_CKPT, f"teacher__{tagged(exp)}.pt")
    if os.path.exists(path):
        return torch.load(path, map_location="cpu")
    teacher = load_trained(exp, method)
    q = encode_corpus(teacher, TOK["train"]["q"], INFER_CHUNKS, (POOLING,))[POOLING]
    c = encode_corpus(teacher, TOK["train"]["c"], INFER_CHUNKS, (POOLING,))[POOLING]
    out = {"q": doc_vectors(*q), "c": doc_vectors(*c)}
    torch.save(out, path)
    del teacher
    gc.collect()
    torch.cuda.empty_cache()
    return out


# Teacher = the strongest fine-tuned full-depth model on dev
candidates = [e for e in ("lorma", "partial", "full") if tagged(e) in RESULTS]
teacher_exp = max(candidates, key=lambda e: RESULTS[tagged(e)]["dev_f1"])
teacher_method = RESULTS[tagged(teacher_exp)]["method"]
print("teacher:", teacher_exp, "| dev F1@7:", round(RESULTS[tagged(teacher_exp)]["dev_f1"], 4))

teacher_tensors = build_teacher_tensors(teacher_exp, teacher_method)
run_experiment("kd_student", "kd", teacher=teacher_tensors)

teacher: full | dev F1@7: 0.2857
v3 missing keys (first 5): ['blocks.0.n1.weight', 'blocks.0.n2.weight', 'blocks.0.attn.q_norm.weight', 'blocks.0.attn.kv_norm.weight', 'blocks.1.n1.weight']
v3 loaded | params=100.1M | from /kaggle/working/phase1_models/legal_lm_mla_moe_1k.pt
kd_student | method=kd | total=56.4M | trainable=56.37M
resumed kd_student from epoch 1


encode:   0%|          | 0/27 [00:00<?, ?it/s]

### Phase 7 Required Experiments & Final Ensemble Evaluation

Below, we set up the final Phase 7 required experiments for the active backbone (`mla_moe`):
- **Full Fine-Tuning** (`full`)
- **LoRMA / Adapter** (`lorma`)
- **KD Student** (`kd_student`)

We run any remaining training epochs, compute metrics on the dev set, and write out final submissions.

In [ ]:
# Execute Phase 7 experiments
for exp, method in [
    ("full", "full"),
    ("lorma", "lorma"),
]:
    print(f"\n=== Running Experiment: {exp} ({method}) ===")
    run_experiment(exp, method)

# Run the Knowledge Distillation (KD) Student experiment
print("\n=== Running KD Student Experiment ===")
run_experiment("kd_student", "kd", teacher=teacher_tensors)

In [ ]:
# Compare results and display a summary table
rows = []
for name, info in RESULTS.items():
    rows.append({
        "Experiment": info.get("exp"),
        "Method": info.get("method"),
        "Total Params (M)": info.get("total_params_M"),
        "Trainable Params (M)": info.get("trainable_M"),
        "Dev F1@7": info.get("dev_f1"),
        "Best Pooling": info.get("best_variant")
    })

if rows:
    df_results = pd.DataFrame(rows)
    display(df_results)
else:
    print("No results recorded yet.")